In [1]:
# iterations
import papermill as pm

# excel consolidation
import os
import pandas as pd
from openpyxl import Workbook
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Alignment
import openpyxl as xl

In [2]:
params_list = [
    {'rule_name': 'fountain', 'param_overbooking': 1, 'sample_protected_pct': 0.5}, 
    {'rule_name': 'fountain', 'param_overbooking': 1, 'sample_protected_pct': 0.1084},
    
    {'rule_name': 'fountain', 'param_overbooking': 3, 'sample_protected_pct': 0.5}, 
    {'rule_name': 'fountain', 'param_overbooking': 3, 'sample_protected_pct': 0.1084},

    {'rule_name': 'fountain', 'param_overbooking': 5, 'sample_protected_pct': 0.5}, 
    {'rule_name': 'fountain', 'param_overbooking': 5, 'sample_protected_pct': 0.1084}, 
    
    {'rule_name': 'fountain', 'param_overbooking': 2, 'sample_protected_pct': 0.5}, 
    {'rule_name': 'fountain', 'param_overbooking': 2, 'sample_protected_pct': 0.1084},

    {'rule_name': 'fountain', 'param_overbooking': 4, 'sample_protected_pct': 0.5}, 
    {'rule_name': 'fountain', 'param_overbooking': 4, 'sample_protected_pct': 0.1084},

    {'rule_name': 'fountain', 'param_overbooking': 6, 'sample_protected_pct': 0.5},
    {'rule_name': 'fountain', 'param_overbooking': 6, 'sample_protected_pct': 0.1084}
]

for params in params_list:
    output_notebook = f'output_pm_nb.ipynb'
    pm.execute_notebook(
        'simulation_process.ipynb', 
        output_notebook,
        parameters=params
    )

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]

In [3]:
verbose = False
rep_iter = '30rep_30iter_10high'
identifier = f"21feb2025_{rep_iter}"

# Set the directory where the Excel files are located
directory = f'/Users/santiospina/Documents/Projects/fair_overbooking/Fair-ML-Overbooking/\
simulation_outputs/excel_files/post_thesis_iterations/{rep_iter}_excel_results'

# Initialize the consolidated Excel file (create a new workbook)
consolidated_path = f'/Users/santiospina/Documents/Projects/fair_overbooking/Fair-ML-Overbooking/\
simulation_outputs/excel_files/post_thesis_iterations/{rep_iter}_excel_results/formatted_consolidated_{identifier}.xlsx'
consolidated_wb = Workbook()

# si el archivo no empieza con ~$ es un archivo que se puede leer
directory_files = [file for file in os.listdir(directory) if file.startswith('fountain') and not file.startswith('~$')]
directory_files = sorted(directory_files)

# Iterate through all filtered files in the directory
for filename in directory_files:
    if filename.endswith(".xlsx"):
        if verbose:
            print(f"Processing file: {filename}")
        # Build the full path for each Excel file
        file_path = os.path.join(directory, filename)
        
        # Extract the sheet name from the filename (between "fountain" and "protected")
        if "fountain" in filename and "protected" in filename:
            sheet_name = filename.split("fountain")[1].split("protected")[0]
        
        # Load the current Excel file
        df = pd.read_excel(file_path, sheet_name=None)  # Read all sheets
        main_sheet_name = list(df.keys())[0]  # Assuming the main sheet is the first sheet
        main_sheet_data = df[main_sheet_name]

        # Add a new sheet to the consolidated workbook if it doesn't exist
        if sheet_name not in consolidated_wb.sheetnames:
            consolidated_wb.create_sheet(sheet_name)
        
        # Get the sheet where data will be copied to
        sheet = consolidated_wb[sheet_name]

        # Find the first empty row in the sheet
        if sheet.max_row > 1:  # If the sheet already has data
            start_row = sheet.max_row + 1
        else:
            start_row = 1  # For the first sheet

        # Copy the data to the new sheet in the consolidated file
        for r_idx, row in main_sheet_data.iterrows():
            for c_idx, value in enumerate(row, 1):
                cell = sheet.cell(row=start_row + r_idx, column=c_idx, value=value)

                # Center the value both horizontally and vertically
                cell.alignment = Alignment(horizontal="center", vertical="center")

        # Apply color fill based on the value in column R
        for row_idx in range(start_row, start_row + len(main_sheet_data)):
            value_in_column_R = sheet[f"R{row_idx}"].value  # Get the value in column R
            if value_in_column_R > 0.05:

                protected_threshold = sheet[f"D{row_idx}"].value
                protected_threshold = round(protected_threshold, 2)
                unprotected_threshold = sheet[f"E{row_idx}"].value
                unprotected_threshold = round(unprotected_threshold, 2)

                if protected_threshold > unprotected_threshold:
                    if verbose:
                        print(f"Protected: {protected_threshold} > {unprotected_threshold} Unprotected: light green")
                    fill = PatternFill(start_color="AEFB7F", end_color="AEFB7F", fill_type="solid") # light green
                    sheet[f"D{row_idx}"].fill = fill
                    sheet[f"E{row_idx}"].fill = fill
                elif protected_threshold < unprotected_threshold:
                    if verbose:
                        print(f"Protected: {protected_threshold} < {unprotected_threshold} Unprotected: salmon")
                    fill = PatternFill(start_color="FA8072", end_color="FA8072", fill_type="solid") # salmon (light red)
                    sheet[f"D{row_idx}"].fill = fill
                    sheet[f"E{row_idx}"].fill = fill
                else:
                    if verbose:
                        print(f"Protected: {protected_threshold} = {unprotected_threshold} Unprotected: light yellow")
                    fill = PatternFill(start_color="FFFD8D", end_color="FFFD8D", fill_type="solid") # light yellow
                    sheet[f"D{row_idx}"].fill = fill
                    sheet[f"E{row_idx}"].fill = fill
                
                # Apply the fill to column R, D, and E for this row
                sheet[f"R{row_idx}"].fill = PatternFill(start_color="CDEEFD", end_color="CDEEFD", fill_type="solid") # light blue
                # sheet[f"D{row_idx}"].fill = fill
                # sheet[f"E{row_idx}"].fill = fill

# HEADERS
for filename in directory_files:
    if filename.endswith(".xlsx"):
        # Build the full path for each Excel file
        file_path = os.path.join(directory, filename)
        
        # Load the current Excel file
        df = pd.read_excel(file_path, sheet_name=None)  # Read all sheets
        main_sheet_name = list(df.keys())[0]  # Assuming the main sheet is the first sheet
        main_sheet_data = df[main_sheet_name]

        # Get the sheet where data was copied to in the consolidated workbook
        sheet = consolidated_wb[filename.split("fountain")[1].split("protected")[0]]

        # Insert the header row (column names) from the original dataframe above the first row of data
        sheet.insert_rows(1)  # Insert a new row at the top
        for col_idx, header in enumerate(main_sheet_data.columns, 1):  # Start at column 1
            cell = sheet.cell(row=1, column=col_idx)  # First row, each column
            cell.value = header  # Set the header name
            cell.font = xl.styles.Font(bold=True)  # Make the header bold
            cell.alignment = Alignment(horizontal="center", vertical="center")  # Center the header text

# Define the columns for merging
merge_columns = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']

# Function to merge cells in the column
def merge_cells(sheet, col_letter):
    row = 2
    while row <= sheet.max_row:
        # Find the last row for the current set of cells that need to be merged
        merge_end_row = row + 4  # Merge in blocks of 5 rows (e.g., A1:A5, A6:A10, A11:A15)
        if merge_end_row > sheet.max_row:  # If the last row exceeds the total number of rows
            merge_end_row = sheet.max_row
        
        # Merge cells in the current block
        sheet.merge_cells(f'{col_letter}{row}:{col_letter}{merge_end_row}')
        
        # Move to the next block of cells
        row = merge_end_row + 1

# Apply the merging to the necessary columns in all sheets
for sheet in consolidated_wb.sheetnames:
    current_sheet = consolidated_wb[sheet]
    for col in merge_columns:
        merge_cells(current_sheet, col)

# After merging, apply centering to all the cells in the sheet
for sheet in consolidated_wb.sheetnames:

    current_sheet = consolidated_wb[sheet]

    for row in current_sheet.iter_rows():
        for cell in row:
            # Center the value both horizontally and vertically
            cell.alignment = Alignment(horizontal="center", vertical="center")

# ancho de las columnas
all_columns = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']
for sheet in consolidated_wb.sheetnames:
    current_sheet = consolidated_wb[sheet]
    for col in all_columns:
        if col == 'H':
            current_sheet.column_dimensions[col].width = 30
        else:
            current_sheet.column_dimensions[col].width = 12

# Save the consolidated Excel file with merged cells, centered values, and colored cells
consolidated_wb.save(consolidated_path)

print("Data consolidation with cell merging, centering, and conditional formatting complete.")

Data consolidation with cell merging, centering, and conditional formatting complete.
